# ✈️ Degradação de Motores Turbofan — Análise Exploratória para Manutenção Preditiva

## Contexto

Motores aeronáuticos operam sob condições extremamente severas, combinando altas temperaturas, pressão, vibração e ciclos contínuos de carga ao longo de sua vida útil. Pequenas alterações em sensores operacionais podem indicar o início de processos de degradação que, se não identificados precocemente, podem evoluir para falhas críticas e elevados custos operacionais.

Neste projeto, utilizamos o dataset de turbofan engines da NASA Prognostics Center of Excellence, amplamente utilizado em estudos de Manutenção Preditiva e Vida Útil Remanescente (RUL). O conjunto contém séries temporais multivariadas de sensores operacionais simulando o comportamento de motores aeronáuticos até o momento de falha.

Mais do que construir modelos preditivos, o objetivo desta análise é compreender o comportamento físico e estatístico da degradação dos motores: como os sensores evoluem ao longo dos ciclos, quais variáveis realmente carregam informação útil e como diferentes regimes operacionais impactam a dinâmica do sistema.

## Perguntas que esta análise responde

**1. Quais sensores realmente apresentam comportamento de degradação?**  
Nem todos os sensores carregam informação útil. Alguns permanecem praticamente constantes ao longo de toda a vida útil do motor, enquanto outros apresentam drift claro conforme a falha se aproxima.

**2. Existem diferentes regimes operacionais no dataset?**  
Certos sensores apresentam distribuições multimodais, sugerindo a existência de múltiplos regimes operacionais influenciando o comportamento do sistema.

**3. Como os sensores evoluem ao longo do ciclo de vida do motor?**  
A degradação ocorre de forma linear, gradual ou abrupta? Alguns sensores só mudam próximos da falha, enquanto outros demonstram sinais precoces de desgaste.

**4. Quais variáveis possuem maior variabilidade e sensibilidade?**  
Sensores com alta variabilidade operacional podem indicar regiões importantes para detecção de anomalias e modelagem de RUL.

**5. Existe correlação forte entre determinados sensores?**  
A relação entre temperatura, pressão, velocidade e fluxo pode revelar dependências físicas importantes e redundâncias no sistema de monitoramento.

**6. Como avaliar a qualidade de cada sensor para prognostics?**  
Além da correlação linear, utilizamos métricas consagradas da literatura PHM (Prognostics and Health Management) para avaliar monotonicidade, prognostibilidade e trendabilidade de cada sensor.

## 0. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.family'] = 'sans-serif'

## 1. Carregamento dos Dados

In [ ]:
COL_NAMES = [
    'unit_id', 'time_cycles',
    'op_setting_1', 'op_setting_2', 'op_setting_3',
    'sensor_temp_fan_inlet',
    'sensor_temp_lpc_outlet',
    'sensor_temp_hpc_outlet',
    'sensor_temp_lpt_outlet',
    'sensor_pressure_inlet',
    'sensor_pressure_fan_inlet',
    'sensor_pressure_ratio',
    'sensor_physical_fan_speed',
    'sensor_physical_core_speed',
    'sensor_engine_pressure_ratio',
    'sensor_static_hpc_outlet',
    'sensor_fuel_flow_ps30',
    'sensor_corrected_fan_speed',
    'sensor_corrected_core_speed',
    'sensor_bypass_ratio',
    'sensor_bleed_enthalpy',
    'sensor_demanded_fan_speed',
    'sensor_demanded_corrected_fan_speed',
    'sensor_hpt_coolant_bleed',
    'sensor_lpt_coolant_bleed',
    'sensor_bpt_ratio'
]

df_train = pd.read_csv('data/train_FD001.txt', sep='\\s+', header=None, names=COL_NAMES)
df_rul   = pd.read_csv('data/RUL_FD001.txt',   header=None, names=['rul'])

print(f'Shape treino: {df_train.shape}')
print(f'Shape RUL:    {df_rul.shape}')
df_train.head()

## 2. Seleção de Features — Remoção de Sensores de Baixa Variância

In [ ]:
DROP_COLS = [
    # Sensores constantes (desvio padrão ≈ 0)
    'sensor_temp_fan_inlet',
    'sensor_pressure_inlet',
    'sensor_pressure_fan_inlet',
    'sensor_physical_fan_speed',
    'sensor_engine_pressure_ratio',
    'sensor_corrected_fan_speed',
    'sensor_bleed_enthalpy',
    'sensor_demanded_corrected_fan_speed',
    'sensor_hpt_coolant_bleed',
    # Configurações operacionais (FD001 = condição operacional única)
    'op_setting_1', 'op_setting_2', 'op_setting_3',
]

df_train = df_train.drop(columns=DROP_COLS)

SENSORS = [col for col in df_train.columns if col.startswith('sensor')]

print(f'Colunas restantes: {df_train.shape[1]}')
print(f'Sensores ativos:   {len(SENSORS)}')
print(SENSORS)

## 3. Estatísticas Descritivas

In [ ]:
print(f"Motores: {df_train['unit_id'].nunique()}")
print(f"Total de linhas: {len(df_train)}\n")

ciclos = df_train.groupby('unit_id')['time_cycles'].max()
print('Ciclos por motor:')
print(ciclos.describe())

In [ ]:
desc = df_train[SENSORS].describe().T
desc['variance'] = df_train[SENSORS].var()
desc['cv'] = (desc['std'] / desc['mean']).abs().round(4)  # Coeficiente de variação
desc = desc.round(3)

# Gradiente de cor: cv alto (vermelho) = mais variação, cv baixo (verde) = mais estável
display(desc.style
    .background_gradient(subset=['cv'], cmap='RdYlGn_r')
    .background_gradient(subset=['std'], cmap='Blues')
    .format(precision=3)
)

| Variável                    |      N |   Média |    DP |     Min |      Q1 | Mediana |      Q3 |     Máx |    Var |    CV |
| :-------------------------- | -----: | ------: | ----: | ------: | ------: | ------: | ------: | ------: | -----: | ----: |
| sensor_temp_lpc_outlet      | 20.631 |  642.68 |  0.50 |  641.21 |  642.33 |  642.64 |  643.00 |  644.53 |   0.25 | 0.001 |
| sensor_temp_hpc_outlet      | 20.631 | 1590.52 |  6.13 | 1571.04 | 1586.26 | 1590.10 | 1594.38 | 1616.91 |  37.59 | 0.004 |
| sensor_temp_lpt_outlet      | 20.631 | 1408.93 |  9.00 | 1382.25 | 1402.36 | 1408.04 | 1414.56 | 1441.49 |  81.01 | 0.006 |
| sensor_pressure_ratio       | 20.631 |  553.37 |  0.89 |  549.85 |  552.81 |  553.44 |  554.01 |  556.06 |   0.78 | 0.002 |
| sensor_physical_core_speed  | 20.631 | 9065.24 | 22.08 | 9021.73 | 9053.10 | 9060.66 | 9069.42 | 9244.59 | 487.65 | 0.002 |
| sensor_static_hpc_outlet    | 20.631 |   47.54 |  0.27 |   46.85 |   47.35 |   47.51 |   47.70 |   48.53 |   0.07 | 0.006 |
| sensor_fuel_flow_ps30       | 20.631 |  521.41 |  0.74 |  518.69 |  520.96 |  521.48 |  521.95 |  523.38 |   0.54 | 0.001 |
| sensor_corrected_core_speed | 20.631 | 8143.75 | 19.08 | 8099.94 | 8133.25 | 8140.54 | 8148.31 | 8293.72 | 363.90 | 0.002 |
| sensor_bypass_ratio         | 20.631 |    8.44 |  0.04 |    8.33 |    8.42 |    8.44 |    8.47 |    8.59 |  0.001 | 0.004 |
| sensor_demanded_fan_speed   | 20.631 |  393.21 |  1.55 |  388.00 |  392.00 |  393.00 |  394.00 |  400.00 |   2.40 | 0.004 |
| sensor_lpt_coolant_bleed    | 20.631 |   38.82 |  0.18 |   38.14 |   38.70 |   38.83 |   38.95 |   39.43 |   0.03 | 0.005 |
| sensor_bpt_ratio            | 20.631 |   23.29 |  0.11 |   22.89 |   23.22 |   23.30 |   23.37 |   23.62 |   0.01 | 0.005 |


### Observações

A tabela resume as principais estatísticas dos 12 sensores ativos após a remoção das variáveis constantes. O coeficiente de variação (cv) — razão entre desvio padrão e média — é o indicador mais relevante aqui: quanto maior, mais o sensor varia ao longo da vida útil do motor.

`sensor_temp_lpt_outlet` e `sensor_static_hpc_outlet` lideram com cv = 0.006, seguidos de `sensor_lpt_coolant_bleed` e `sensor_bpt_ratio` (cv = 0.005). São esses os sensores que mais "se movem" ao longo da degradação — e que, como veremos adiante, terão maior poder preditivo.

Na outra ponta, `sensor_temp_lpc_outlet` e `sensor_fuel_flow_ps30` têm cv próximo de zero — variação pequena em termos relativos, embora ainda presente em termos absolutos.

Vale notar que os valores absolutos das médias são muito diferentes entre sensores (de 8 para o bypass_ratio a 9065 para a velocidade do núcleo) — o que reforça a necessidade de normalização antes da modelagem.

**Próximo passo → Seção 4: Distribuição dos Sensores**

Com as estatísticas gerais mapeadas, visualizamos agora a forma de cada distribuição — identificando assimetrias, multimodalidade e comportamentos atípicos que as médias e desvios padrão não capturam.

## 4. Distribuição dos Sensores

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(14, 7))
axes = axes.flatten()

for i, sensor in enumerate(SENSORS):
    short = sensor.replace('sensor_', '')
    sns.kdeplot(df_train[sensor], ax=axes[i], fill=True,
                color='steelblue', alpha=0.6, linewidth=1.5)
    axes[i].set_title(short, fontsize=10, fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Densidade', fontsize=8)
    axes[i].tick_params(labelsize=7)

for j in range(len(SENSORS), len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Distribuição dos Sensores — CMAPSS FD001', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

![Distribuição dos Sensores](midia/distribuicao.png)

### Observações  — Seção 4

A maior parte dos sensores apresenta distribuições relativamente contínuas e estáveis, ainda que com diferentes níveis de dispersão e assimetria. `physical_core_speed` e `corrected_core_speed` exibem *right skew* — cauda longa à direita — sugerindo leituras menos frequentes em regimes de alta velocidade, possivelmente associadas à degradação avançada do motor.

`demanded_fan_speed` se destaca dos demais ao apresentar comportamento claramente multimodal, formando múltiplos picos bem definidos na distribuição — hipótese investigada na próxima etapa.



### 4.1 Investigação Multimodal — demanded_fan_speed

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.kdeplot(df_train['sensor_demanded_fan_speed'],
            ax=axes[0], fill=True, color='steelblue',
            alpha=0.6, linewidth=2, bw_adjust=0.5)
axes[0].set_title('demanded_fan_speed — KDE', fontweight='bold')

axes[1].hist(df_train['sensor_demanded_fan_speed'],
             bins=80, color='steelblue', alpha=0.7, edgecolor='white')
axes[1].set_title('demanded_fan_speed — Histograma', fontweight='bold')

fig.suptitle('Investigação Multimodal — demanded_fan_speed', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Valores únicos: {df_train['sensor_demanded_fan_speed'].nunique()}")
print(f"\nValores únicos por motor:")
print(df_train.groupby('unit_id')['sensor_demanded_fan_speed'].nunique().describe())

![Investigação Multimodal](midia/invest_mult.png)

### Observações — Seção 4.1

O sensor `demanded_fan_speed` não é uma medição contínua — é uma variável de **setpoint discreto** com apenas 13 valores inteiros possíveis. Isso significa que ele não deve ser tratado como os demais sensores em etapas de normalização.

**Próximo passo → Seção 5: Boxplots por Sensor**

Com a forma das distribuições já mapeada, os boxplots vão complementar a análise mostrando outliers e comparando a dispersão de todos os sensores lado a lado.

## 5. Boxplots dos Sensores

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(14, 8))
axes = axes.flatten()

for i, sensor in enumerate(SENSORS):
    short = sensor.replace('sensor_', '')
    sns.boxplot(
        y=df_train[sensor].dropna(),
        ax=axes[i],
        color='steelblue',
        width=0.5,
        linewidth=1.2,
        flierprops=dict(marker='o', markersize=3, alpha=0.3, color='steelblue'),
        medianprops=dict(color='darkred', linewidth=2),
        boxprops=dict(alpha=0.6),
        whiskerprops=dict(linewidth=1.2),
        capprops=dict(linewidth=1.2)
    )
    axes[i].set_title(short, fontsize=10, fontweight='bold')
    axes[i].set_xticks([])
    axes[i].tick_params(labelsize=7)
    axes[i].spines[['top', 'right']].set_visible(False)
    axes[i].set_ylabel('')  

for j in range(len(SENSORS), len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Boxplots dos Sensores — CMAPSS FD001', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('sensor_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

![Boxplots dos Sensores](midia/boxplots.png)

### Observações — Seção 5
Os boxplots revelam três padrões distintos:

- **Outliers superiores com right skew:** `physical_core_speed` e `corrected_core_speed` têm cauda longa à direita, com muitos outliers acima do whisker superior. `static_hpc_outlet` apresenta comportamento similar.

- **Distribuições compactas:** `temp_lpc_outlet`, `pressure_ratio` e `fuel_flow_ps30` têm IQR pequeno e poucos outliers, indicando sensores estáveis e bem comportados.

- **`demanded_fan_speed`:** boxplot compacto com outliers isolados em ambos os lados, coerente com a natureza discreta já identificada na seção anterior.

**Próximo passo → Seção 6: Matriz de Correlação entre Sensores**

Com a distribuição individual de cada sensor mapeada, o próximo passo é entender as relações *entre* eles. Sensores altamente correlacionados carregam informação redundante — identificá-los agora evita multicolinearidade no modelo.

## 6. Matriz de Correlação entre Sensores

In [ ]:
short_names = {col: col.replace('sensor_', '') for col in SENSORS}
corr = df_train[SENSORS].corr().rename(index=short_names, columns=short_names)

fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(
    corr,
    annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    linewidths=0.4, linecolor='white',
    annot_kws={'size': 7, 'weight': 'bold'},
    square=True,
    cbar_kws={'shrink': 0.8, 'label': 'Pearson r'},
    ax=ax
)
ax.set_title('Matriz de Correlação dos Sensores — CMAPSS FD001', fontsize=13, fontweight='bold', pad=15)
ax.tick_params(axis='both', labelsize=8)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

![Matriz de Correlação](midia/matriz.png)

### Observações — Seção 6

A matriz revela dois grupos de sensores altamente correlacionados entre si:

**Grupo térmico-aerodinâmico** (`temp_lpt_outlet`, `static_hpc_outlet`, `bypass_ratio`, `fuel_flow_ps30`, `pressure_ratio`): correlações acima de |0.75| entre si, formando um bloco coeso. Esses sensores capturam o mesmo fenômeno físico — o aquecimento progressivo do motor — por ângulos diferentes.

**Grupo de velocidade** (`physical_core_speed`, `corrected_core_speed`): correlação de 0.96 entre si, praticamente redundantes. Manter ambos no modelo adiciona pouca informação nova.

**Sensores mais independentes:** `physical_core_speed` e `corrected_core_speed` têm correlações baixas com o restante (~0.2–0.3), sugerindo que capturam um fenômeno distinto dos demais.

Alta correlação entre sensores não significa que devemos descartá-los agora, isso é decisão de feature selection, que será feita com base nas métricas PHM mais adiante. O que a matriz nos diz é que o modelo precisará lidar com multicolinearidade.

**Próximo passo → Seção 7: Vida Útil dos Motores**

Antes de partir para a análise de degradação, é necessário entender a distribuição dos ciclos de vida entre os 100 motores, quantos ciclos em média, qual a variabilidade, e se existem outliers que possam distorcer o target RUL.

![Vida Útil dos Motores](midia/vidautil.png)

## 7. Vida Útil dos Motores

In [ ]:
vida_util = df_train.groupby('unit_id')['time_cycles'].max()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(vida_util, bins=20, color='steelblue', alpha=0.7, edgecolor='white')
axes[0].axvline(vida_util.mean(),   color='darkred', linewidth=2, linestyle='--', label=f'Média: {vida_util.mean():.0f}')
axes[0].axvline(vida_util.median(), color='orange',  linewidth=2, linestyle='--', label=f'Mediana: {vida_util.median():.0f}')
axes[0].set_title('Distribuição da Vida Útil dos Motores', fontweight='bold')
axes[0].set_xlabel('Ciclos até a Falha')
axes[0].set_ylabel('Contagem')
axes[0].legend()

axes[1].boxplot(vida_util, vert=False, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6),
                medianprops=dict(color='darkred', linewidth=2),
                flierprops=dict(marker='o', markersize=5, alpha=0.5))
axes[1].set_title('Vida Útil dos Motores — Boxplot', fontweight='bold')
axes[1].set_xlabel('Ciclos até a Falha')
axes[1].set_yticks([])

fig.suptitle('Vida Útil dos Motores — CMAPSS FD001', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('engine_lifetime.png', dpi=150, bbox_inches='tight')
plt.show()

### Observações — Seção 7

Os 100 motores têm vida útil média de **206 ciclos** (mediana 199), com distribuição com *right skew* moderado — a maioria falha entre 150 e 230 ciclos, mas há uma cauda de motores que duram significativamente mais.

O boxplot evidencia **3 outliers claros acima de 300 ciclos**, com um chegando próximo de 360. Esses motores têm vida útil quase o dobro da mediana, o que tem implicação direta na construção do target RUL: sem tratamento, o modelo tentará aprender a diferença entre "RUL = 200" e "RUL = 360" em motores saudáveis — uma distinção que operacionalmente não tem significado.

Isso motiva o uso de um **RUL cap** nas etapas de pré-processamento — tipicamente 125 ciclos na literatura CMAPSS. Essa técnica será aplicada futuramente, na etapa de preparação dos dados para o modelo.

**Próximo passo → Seção 8: Criação do Target RUL**

Com a distribuição de vida útil compreendida, calculamos o RUL linear para cada observação — base para todas as análises de degradação que vêm a seguir.

## 8. Criação do Target RUL

In [ ]:
# Para cada motor, identificamos o ciclo em que ele falhou
rul_max = df_train.groupby('unit_id')['time_cycles'].max().reset_index()
rul_max.columns = ['unit_id', 'max_cycles']

# Juntamos essa informação no dataframe principal
df_train = df_train.merge(rul_max, on='unit_id')

# RUL = ciclos restantes até a falha (target linear)
df_train['RUL'] = df_train['max_cycles'] - df_train['time_cycles']
df_train = df_train.drop(columns='max_cycles')

print(f'Shape final: {df_train.shape}')
df_train[['unit_id', 'time_cycles', 'RUL']].head(10)

## 9. Degradação Temporal dos Sensores

In [ ]:
degradation = df_train.groupby('RUL')[SENSORS].mean().reset_index().sort_values('RUL', ascending=False)

fig, axes = plt.subplots(4, 3, figsize=(16, 7.5))
axes = axes.flatten()

for i, sensor in enumerate(SENSORS):
    short = sensor.replace('sensor_', '')
    rolling = degradation[sensor].rolling(window=10, center=True).mean()

    # Sinal bruto mais suave atrás
    axes[i].plot(degradation['RUL'], degradation[sensor],
                 color='steelblue', alpha=0.4, linewidth=0.8)
    # Tendência por cima
    axes[i].plot(degradation['RUL'], rolling,
                 color='darkred', linewidth=2.0)

    axes[i].set_title(short, fontsize=9, fontweight='bold', loc='left', pad=4)
    axes[i].invert_xaxis()
    axes[i].tick_params(labelsize=7)
    axes[i].spines[['top', 'right']].set_visible(False)
    axes[i].set_ylabel('', labelpad=0)

for j in range(len(SENSORS), len(axes)):
    axes[j].set_visible(False)

# Label do eixo X só nos subplots da última linha
for ax in axes[9:12]:
    ax.set_xlabel('RUL', fontsize=8)

fig.suptitle('Degradação Temporal dos Sensores — CMAPSS FD001',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('degradation_all.png', dpi=150, bbox_inches='tight')
plt.show()

![Degradação Temporal](midia/degradacao.png)

### Observações — Seção 9

Os plots revelam três comportamentos distintos ao longo da vida do motor:

**Crescimento exponencial próximo à falha (RUL < 50):**  
`temp_lpc_outlet`, `temp_hpc_outlet`, `temp_lpt_outlet`, `static_hpc_outlet`, `physical_core_speed`, `corrected_core_speed`, `bypass_ratio` e `demanded_fan_speed` — todos relativamente estáveis durante a maior parte da vida útil, mas com aceleração clara nos últimos ciclos. Esse padrão exponencial é exatamente o que modelos como LSTM capturam bem.

**Queda exponencial próximo à falha:**  
`pressure_ratio`, `fuel_flow_ps30` e `lpt_coolant_bleed` — comportamento inverso, mas igualmente informativo. O motor perde eficiência de pressão e fluxo de combustível conforme se degrada.

**Ponto importante:** a degradação visível é resultado do comportamento coletivo dos 100 motores. A variabilidade individual entre motores será investigada na análise de trendabilidade.

**Próximo passo → Seção 10: Correlação dos Sensores com o RUL**

Com os padrões de degradação mapeados visualmente, quantificamos agora a força da relação linear entre cada sensor e o RUL.

## 10. Correlação dos Sensores com o RUL

In [ ]:
corr_rul = df_train[SENSORS + ['RUL']].corr()['RUL'].drop('RUL').sort_values()

fig, ax = plt.subplots(figsize=(10, 6))

# Azul = correlação positiva (sensor cai com degradação)
# Vermelho = correlação negativa (sensor sobe com degradação)
colors = ['darkred' if v < 0 else 'steelblue' for v in corr_rul.values]
bars = ax.barh(corr_rul.index.str.replace('sensor_', ''), corr_rul.values, color=colors, alpha=0.8)

for bar, val in zip(bars, corr_rul.values):
    ax.text(val + (0.005 if val >= 0 else -0.005), bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', ha='left' if val >= 0 else 'right', fontsize=9)

ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Correlação de Pearson com o RUL', fontsize=11)
ax.set_title('Correlação dos Sensores com o RUL — CMAPSS FD001', fontsize=13, fontweight='bold')
ax.tick_params(labelsize=9)
plt.tight_layout()
plt.savefig('corr_with_rul.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nCorrelação com RUL:')
print(corr_rul.round(3))

![Correlação com RUL](midia/correlacao.png)

### Observações — Seção 10
O gráfico quantifica o que já observamos visualmente na seção anterior:

**Sensores mais preditivos (correlação negativa forte):**  
`static_hpc_outlet` (-0.696) e `temp_lpt_outlet` (-0.679) lideram — aumentam consistentemente conforme o motor se degrada, sendo os candidatos mais fortes para o modelo.

**Sensores com correlação positiva forte:**  
`fuel_flow_ps30` (0.672) e `pressure_ratio` (0.657) — diminuem com a degradação, mas com força preditiva equivalente aos negativos.

**Sensores mais fracos linearmente:**  
`corrected_core_speed` (-0.307) e `physical_core_speed` (-0.390) têm a menor correlação linear. No entanto, vimos nos plots de degradação que ambos apresentam crescimento exponencial acentuado nos últimos ciclos — um modelo não-linear como LSTM consegue capturar esse padrão mesmo com Pearson baixo.

A correlação de Pearson é apenas um primeiro filtro — mede relações lineares e não captura a aceleração exponencial característica da degradação. A avaliação completa de cada sensor será feita com métricas PHM na próxima seção.


Passamos agora para uma avaliação mais robusta, usando métricas consagradas na literatura de manutenção preditiva que vão além da correlação linear.

## 11. Métricas PHM — Monotonicidade e Prognostibilidade 

Estas duas métricas vêm da literatura IEEE PHM (Prognostics and Health Management) e avaliam a qualidade de cada sensor para predição de RUL:

- **Monotonicidade:** o sensor caminha em uma única direção conforme o motor degrada? Um sensor que oscila aleatoriamente carrega pouco sinal preditivo.
- **Prognostibilidade:** o sensor converge para um valor similar no momento da falha em todos os motores? Alta prognostibilidade significa que o modelo consegue aprender um "ponto de ruptura" consistente.


In [ ]:
phm_metrics = []

for sensor in SENSORS:
    monotonicities = []
    initial_values = []
    final_values   = []

    for unit, group in df_train.groupby('unit_id'):
        group_sorted = group.sort_values('time_cycles')

        # Correlação de Spearman entre leitura do sensor e tempo:
        # captura tendências monotônicas mesmo não-lineares (ao contrário de Pearson).
        # Usamos o valor absoluto pois tanto subida quanto descida consistente é informativo.
        rho, _ = spearmanr(group_sorted['time_cycles'], group_sorted[sensor])
        if not np.isnan(rho):
            monotonicities.append(abs(rho))

        # Capturamos o valor do sensor no início e no momento da falha
        # para quantificar quanto — e quão consistentemente — ele muda
        initial_values.append(group_sorted[sensor].iloc[0])
        final_values.append(group_sorted[sensor].iloc[-1])

    mean_monotonicity = np.mean(monotonicities)

    # Fórmula de Prognostibilidade (padrão IEEE PHM):
    #   exp( -std(final) / |média(final) - média(inicial)| )
    # Numerador  → dispersão do sensor no momento da falha entre os motores
    # Denominador → deslocamento médio do início até a falha
    # Resultado em [0, 1]: 1 = valor de falha idêntico para todos os motores
    std_final    = np.std(final_values)
    mean_diff    = abs(np.mean(final_values) - np.mean(initial_values))
    prognostability = np.exp(-std_final / mean_diff) if mean_diff != 0 else 0

    phm_metrics.append({
        'Sensor':          sensor.replace('sensor_', ''),
        'Monotonicidade':  mean_monotonicity,
        'Prognostibilidade': prognostability
    })

df_phm = (pd.DataFrame(phm_metrics)
            .sort_values(by='Prognostibilidade', ascending=False)
            .reset_index(drop=True))

# --- Plot 1: Monotonicidade vs Prognostibilidade ---
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

sns.barplot(data=df_phm.sort_values('Monotonicidade', ascending=False),
            y='Sensor', x='Monotonicidade',
            ax=axes[0], palette='Blues_r', edgecolor='black', alpha=0.8)
axes[0].set_title('Monotonicidade dos Sensores (quanto maior, melhor)', fontweight='bold', fontsize=12)
axes[0].set_xlabel('Correlação de Spearman Absoluta Média')
axes[0].grid(True, axis='x', linestyle='--', alpha=0.4)

sns.barplot(data=df_phm,
            y='Sensor', x='Prognostibilidade',
            ax=axes[1], palette='Oranges_r', edgecolor='black', alpha=0.8)
axes[1].set_title('Prognostibilidade dos Sensores (quanto maior, melhor)', fontweight='bold', fontsize=12)
axes[1].set_xlabel('Score de Prognostibilidade [0 a 1]')
axes[1].grid(True, axis='x', linestyle='--', alpha=0.4)

fig.suptitle('Métricas PHM — CMAPSS FD001', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('phm_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

![Métricas PHM](midia/metricas.png)

### Observações — Seção 11: Métricas PHM

Os resultados confirmam e enriquecem o que a correlação de Pearson havia sugerido:

**Melhores sensores (alto em ambas as métricas):**
`static_hpc_outlet`, `temp_lpt_outlet`, `fuel_flow_ps30` e `bypass_ratio` se destacam com scores elevados tanto em monotonicidade quanto em prognostibilidade — são os candidatos mais sólidos para o modelo preditivo.

**Caso interessante — `corrected_core_speed` e `physical_core_speed`:**
Apesar de apresentarem monotonicidade razoável (~0.65), a prognostibilidade é a mais baixa de todos os sensores (~0.20–0.25). Isso significa que, embora o sensor caminhe em uma direção consistente ao longo da vida do motor, o valor que ele assume no momento da falha varia muito entre os 100 motores — tornando difícil para o modelo aprender um "ponto de ruptura" confiável.

**`temp_hpc_outlet`:**
Menor monotonicidade do grupo, mas prognostibilidade razoável — o sensor oscila durante a vida útil mas converge a um valor mais previsível na falha.

De forma geral, os sensores térmicos e aerodinâmicos dominam ambas as métricas, reforçando que a degradação do turbofan se manifesta principalmente como um fenômeno de aquecimento e perda de eficiência de pressão.

---



### Conclusão Geral da EDA

Esta análise exploratória estabelece uma base sólida para as próximas etapas do projeto. Os principais achados foram:

- **9 sensores com variância zero** foram removidos, restando 12 sensores ativos
- A degradação segue um **padrão exponencial nos últimos ~50 ciclos** para a maioria dos sensores térmicos
- Dois grupos de sensores altamente correlacionados foram identificados, sinalizando **multicolinearidade** a ser tratada na modelagem
- As métricas PHM apontam `static_hpc_outlet`, `temp_lpt_outlet` e `fuel_flow_ps30` como os **sensores mais promissores** para predição de RUL

**Próximo notebook → Análise de Sobrevivência**

Antes da modelagem preditiva, a análise de sobrevivência (Kaplan-Meier, Weibull) fornecerá um entendimento probabilístico da distribuição do tempo até a falha — estabelecendo um baseline estatístico robusto para comparação com os modelos de machine learning.
